In [1]:
# Single-cell test for moltie/llm/client.py : verify_with_ollama()

from pathlib import Path
import sys, textwrap

CODE = Path("/home/hello/Projects/Statements/code").resolve()
PDF_PATH = Path("/home/hello/Projects/Statements/code/appeals/SAINSBURYS SUPERMARKETS LIMITED_vs_HITT.pdf").resolve()
assert CODE.exists(), CODE
assert PDF_PATH.exists(), PDF_PATH
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

# --- import only what we need ---
from moltie.llm.client import verify_with_ollama, LLMClientConfig
from moltie.schemas.verdict import Verdict
from moltie.agent.retrieve import retrieve_windowed_evidence
from moltie.corpus.chunker import iter_paragraphs
from moltie.schemas.query_object import AtomQuery

# --- PDF -> text fixture ---
try:
    from pypdf import PdfReader
except Exception:
    from PyPDF2 import PdfReader

text = "\n\n".join([(p.extract_text() or "") for p in PdfReader(str(PDF_PATH)).pages]).strip()
assert len(text) > 500, "PDF text extraction looks empty/too short (likely scanned OCR)."

doc_id = PDF_PATH.stem

# --- build paras [{para_id,text}] ---
paras = [{"para_id": f"p{i:05d}", "text": ptxt} for i, (_, _, ptxt) in enumerate(iter_paragraphs(text), start=1)]
assert len(paras) > 0

# --- build AtomQuery without guessing constructor fields ---
ann = getattr(AtomQuery, "__annotations__", {})
kwargs = {}
for k in ann.keys():
    if k == "atom_id": kwargs[k] = "X_TEST"
    elif k == "x_tests": kwargs[k] = ["X1"]
    elif k == "proposition": kwargs[k] = "Substitution / reasonableness test in unfair dismissal"
    elif k == "positive_indicators": kwargs[k] = ["substitute", "reasonable", "investigation", "unfair dismissal", "tribunal"]
    elif k in ("excludes", "keyword_seeds", "expansion_terms"): kwargs[k] = []
    else: kwargs[k] = None
atom = AtomQuery(**kwargs)

# --- retrieve evidence pack (proven working) ---
rr = retrieve_windowed_evidence(doc_id=doc_id, paras=paras, atom=atom, k=8, min_hits=2, window_size=24, stride=12, top_windows=2)
assert len(rr.paras) > 0

# --- build a minimal strict prompt (no verifier_prompt.py needed for this test) ---
# IMPORTANT: embed atom_id/doc_id as JSON-like keys so _extract_from_prompt can recover them.
evidence_lines = []
MAX_PARA_CHARS = 900
for p in rr.paras:
    txt = (p["text"] or "").strip().replace("\n", " ")
    if len(txt) > MAX_PARA_CHARS:
        txt = txt[:MAX_PARA_CHARS] + "…"
    evidence_lines.append(f'{p["para_id"]}: {txt}')

evidence_blob = "\n\n".join(evidence_lines)  # ✅ add this

prompt = textwrap.dedent(f"""
You are a legal reasoning engine.

OUTPUT CONTRACT (NON-NEGOTIABLE):
- Output ONE SINGLE JSON object only. No prose, no markdown.
- Use EXACTLY these top-level keys (all required, no extras):
  atom_id, doc_id, relevant, matched_X, precedent_score, confidence, anchors,
  use_mode, proposition_winner, appeal_outcome, successful_party,
  distinguishers, note, retrieval_score, retrieval_method
- matched_X MUST be a list of X-test IDs only, chosen from: ["X1","X2","X3","X4","X5"].
  NEVER put keywords in matched_X.
  For this run, ONLY use: ["X1"] (or []).

- anchors MUST be a list of objects with EXACT keys:
  para_id, quote, why_it_matters

- IMPORTANT: You may ONLY quote from the Evidence paragraphs below.
  The quote MUST be a verbatim substring of that paragraph text.
  The para_id MUST be one of the provided para_id values.

- HARD LIMITS:
  * max 2 anchors
  * each quote <= 240 characters
  * each why_it_matters <= 140 characters
  * note = one sentence <= 160 characters

- If you cannot provide at least 1 valid anchor, set:
  relevant=false, anchors=[], matched_X=[], precedent_score=0, confidence=0,
  use_mode="contrast", proposition_winner="unclear", appeal_outcome="unknown",
  successful_party="unclear", distinguishers=[], note="No relevant information found.",
  retrieval_score=null, retrieval_method=null

"atom_id": "{atom.atom_id}"
"doc_id": "{doc_id}"

Proposition:
{atom.proposition}

Positive indicators:
- {chr(10).join(atom.positive_indicators or [])}

Evidence paragraphs (ONLY source for anchors):
{evidence_blob}
""").strip()

# --- call client ---
cfg = LLMClientConfig(
    model="mistral-small3.2:latest",
    ollama_url="http://localhost:11434/api/generate",
    timeout_s=180,
    temperature=0.0,
    num_predict=1200,
    max_retries=1,   # keep test tight
)

v = verify_with_ollama(prompt, cfg)

# --- assertions ---
assert isinstance(v, Verdict), type(v)
assert v.atom_id == atom.atom_id, (v.atom_id, atom.atom_id)
assert v.doc_id == doc_id, (v.doc_id, doc_id)

# schema-level invariants already enforced, but let's print proof
print("OK: got Verdict")
print("relevant:", v.relevant, "use_mode:", v.use_mode, "score/conf:", v.precedent_score, v.confidence)
print("anchors:", len(v.anchors))
if v.anchors:
    print("anchor[0]:", v.anchors[0].para_id)
    print("quote head:", v.anchors[0].quote[:140])
    # extra sanity: anchor para_id is in our evidence pack
    ev_ids = {p["para_id"] for p in rr.paras}
    assert v.anchors[0].para_id in ev_ids, f"Anchor para_id not in evidence pack: {v.anchors[0].para_id}"


OK: got Verdict
relevant: True use_mode: contrast score/conf: 85 90
anchors: 2
anchor[0]: p00024
quote head: The employer, not the tribunal, is the proper person to conduct the investigation into the alleged misconduct. The function of the tribunal 


In [ ]:
from pathlib import Path
import sys, importlib, inspect

CODE = Path("/home/hello/Projects/Statements/code").resolve()
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))

import moltie.llm.client as client
importlib.reload(client)

src = inspect.getsource(client._extract_json_object)
print(src)
